In [26]:
!pip install transformers datasets torch --quiet

In [27]:
import torch
import math
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GPT2LMHeadModel.from_pretrained("gpt2")
model = model.to(device)
model.eval()

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [28]:
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")

In [29]:
text = "\n\n".join(dataset["text"])

In [30]:
encodings = tokenizer(text, return_tensors="pt")
input_ids = encodings.input_ids.to(device)

Token indices sequence length is longer than the specified maximum sequence length for this model (251048 > 1024). Running this sequence through the model will result in indexing errors


In [31]:
stride = 512
max_length = model.config.n_positions

nlls = []
seq_len = input_ids.size(1)

for i in range(0, seq_len, stride):
    begin_loc = max(i + stride - max_length, 0)
    end_loc = min(i + stride, seq_len)
    trg_len = end_loc - i
    
    input_ids_slice = input_ids[:, begin_loc:end_loc]
    target_ids = input_ids_slice.clone()
    target_ids[:, :-trg_len] = -100

    with torch.no_grad():
        outputs = model(input_ids_slice, labels=target_ids)
        neg_log_likelihood = outputs.loss * trg_len

    nlls.append(neg_log_likelihood)

ppl = torch.exp(torch.stack(nlls).sum() / seq_len)
print("Baseline Perplexity:", ppl.item())

Baseline Perplexity: 26.392776489257812


In [33]:
block = model.transformer.h[0]

print("c_fc weight shape:", block.mlp.c_fc.weight.shape)
print("c_fc bias shape:", block.mlp.c_fc.bias.shape)

print("c_proj weight shape:", block.mlp.c_proj.weight.shape)
print("c_proj bias shape:", block.mlp.c_proj.bias.shape)

c_fc weight shape: torch.Size([768, 3072])
c_fc bias shape: torch.Size([3072])
c_proj weight shape: torch.Size([3072, 768])
c_proj bias shape: torch.Size([768])


In [34]:
block = model.transformer.h[0]

fc_weight = block.mlp.c_fc.weight.data
proj_weight = block.mlp.c_proj.weight.data

fc_correct = fc_weight.T
proj_correct = proj_weight.T

print("True c_fc shape:", fc_correct.shape)
print("True c_proj shape:", proj_correct.shape)

True c_fc shape: torch.Size([3072, 768])
True c_proj shape: torch.Size([768, 3072])
